In [2]:
# Step 1: Imports 
import numpy as np
import pandas as pd
import json
import os
import re
import string
import pickle
import matplotlib.pyplot as plt
from collections import Counter

# Importing all the tensorflow libraries
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import seaborn as sns

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


In [3]:
df = pd.read_csv("drug_reviews_dataset.csv")

print("Total rows:", len(df))
df.head()


Total rows: 600


,drug_name,condition,review_text,rating,sentiment_label,side_effects_present,side_effect_severity,side_effects_text
0,Trazodone,Insomnia,Trazodone is working well for my Insomnia. I'm...,8,Positive,Yes,Moderate,"hot flashes, night sweats"
1,Enoxaparin,Deep Vein Thrombosis,10/10 for Enoxaparin. My Deep Vein Thrombosis ...,10,Positive,No,NaN,NaN
2,Risperidone,Bipolar Disorder,Doctor put me on Risperidone for my Bipolar Di...,9,Positive,Yes,Moderate,mood swings
3,Alendronate,Osteoporosis,My doctor prescribed Alendronate for my Osteop...,3,Negative,No,NaN,NaN
4,Allopurinol,Gout,Honestly wasn't expecting much from Allopurino...,8,Positive,Yes,Mild,mild weight gain


In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

In [8]:
#  Step 4: Text cleaning - NLP 
def clean_text(text):
    # Convert to lower case 
    text = str(text).lower()
    # remove all punctuations
    text = text.translate(str.maketrans('', '', string.punctuation))
    # remove all the extra spaces 
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Applying this clean text function to both train and test data 
train_df["text"] = train_df["review_text"].apply(clean_text)
test_df["text"]  = test_df["review_text"].apply(clean_text)


In [9]:
# Step 5: Label encoding  
# SENTIMENT
le_sent = LabelEncoder()
y_sent_train = le_sent.fit_transform(train_df["sentiment_label"])
y_sent_test  = le_sent.transform(test_df["sentiment_label"])

# SIDE EFFECT
le_side = LabelEncoder()
y_side_train = le_side.fit_transform(train_df["side_effects_present"])
y_side_test  = le_side.transform(test_df["side_effects_present"])

# SEVERITY
le_sev = LabelEncoder()
y_sev_train = le_sev.fit_transform(train_df["side_effect_severity"])
y_sev_test  = le_sev.transform(test_df["side_effect_severity"])


In [10]:
# Step 6: Tokenisation & padding 

MAX_WORDS = 20000   # vocabulary size
MAX_LEN   = 120     # max tokens per sample

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

X_train = pad_sequences(
    tokenizer.texts_to_sequences(train_df["text"]),
    maxlen=MAX_LEN, padding="post", truncating="post"
)
X_test = pad_sequences(
    tokenizer.texts_to_sequences(test_df["text"]),
    maxlen=MAX_LEN, padding="post", truncating="post"
)

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

In [16]:
# Step 8: Bidirectional LSTM model 
EMBEDDING_DIM = 128

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input

input_layer = Input(shape=(MAX_LEN,))

x = Embedding(vocab_size, 128)(input_layer)
x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Dropout(0.4)(x)
x = Bidirectional(LSTM(64))(x)
x = Dense(128, activation="relu")(x)

# 🔥 MULTI HEAD OUTPUTS
sentiment_output = Dense(3, activation="softmax", name="sentiment")(x)
side_output = Dense(2, activation="softmax", name="side")(x)
severity_output = Dense(4, activation="softmax", name="severity")(x)

model = Model(
    inputs=input_layer,
    outputs=[sentiment_output, side_output, severity_output]
)


model.compile(
    optimizer=Adam(1e-3),
    loss={
        "sentiment": "sparse_categorical_crossentropy",
        "side": "sparse_categorical_crossentropy",
        "severity": "sparse_categorical_crossentropy"
    },
    metrics={
        "sentiment": "accuracy",
        "side": "accuracy",
        "severity": "accuracy"
    }
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 120)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 120, 128)  │    127,104 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ (None, 120, 256)  │    263,168 │ embedding_2[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 120, 256)  │          0 │ bidirectional_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_5     │ (None, 128)       │    164,352 │ dropout_2[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     16,512 │ bidirectional_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sentiment (Dense)   │ (None, 3)         │        387 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side (Dense)        │ (None, 2)         │        258 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ severity (Dense)    │ (None, 4)         │        516 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 572,297 (2.18 MB)

 Trainable params: 572,297 (2.18 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Step 9: Train the model 
os.makedirs("model", exist_ok=True)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy"),
    ModelCheckpoint("model/symptom_model1.h5", save_best_only=True, monitor="val_accuracy"),
    ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-6),
]

history = model.fit(
    X_train,
    {
        "sentiment": y_sent_train,
        "side": y_side_train,
        "severity": y_sev_train
    },
    validation_data=(
        X_test,
        {
            "sentiment": y_sent_test,
            "side": y_side_test,
            "severity": y_sev_test
        }
    ),
    epochs=20,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - loss: 3.1343 - sentiment_accuracy: 0.3881 - sentiment_loss: 1.0954 - severity_accuracy: 0.3819 - severity_loss: 1.3594 - side_accuracy: 0.6169 - side_loss: 0.6796

c:\Users\Lenovo\anaconda3\Lib\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_accuracy` which is not available. Available metrics are: loss,sentiment_accuracy,sentiment_loss,severity_accuracy,severity_loss,side_accuracy,side_loss,val_loss,val_sentiment_accuracy,val_sentiment_loss,val_severity_accuracy,val_severity_loss,val_side_accuracy,val_side_loss
  current = self.get_monitor_value(logs)
c:\Users\Lenovo\anaconda3\Lib\site-packages\keras\src\callbacks\model_checkpoint.py:276: UserWarning: Can save best model only with val_accuracy available.
  if self._should_save_model(epoch, batch, logs, filepath):


15/15 ━━━━━━━━━━━━━━━━━━━━ 8s 202ms/step - loss: 3.0956 - sentiment_accuracy: 0.3896 - sentiment_loss: 1.0901 - severity_accuracy: 0.4208 - severity_loss: 1.3224 - side_accuracy: 0.5854 - side_loss: 0.6831 - val_loss: 3.0765 - val_sentiment_accuracy: 0.4500 - val_sentiment_loss: 1.0584 - val_severity_accuracy: 0.4000 - val_severity_loss: 1.3370 - val_side_accuracy: 0.5750 - val_side_loss: 0.6718 - learning_rate: 0.0010
Epoch 2/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 445ms/step - loss: 2.9046 - sentiment_accuracy: 0.6086 - sentiment_loss: 1.0090 - severity_accuracy: 0.4246 - severity_loss: 1.2740 - side_accuracy: 0.7376 - side_loss: 0.6216

15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 519ms/step - loss: 2.7981 - sentiment_accuracy: 0.6271 - sentiment_loss: 0.9585 - severity_accuracy: 0.4417 - severity_loss: 1.2449 - side_accuracy: 0.7542 - side_loss: 0.5947 - val_loss: 2.2757 - val_sentiment_accuracy: 0.6667 - val_sentiment_loss: 0.8087 - val_severity_accuracy: 0.5417 - val_severity_loss: 1.0830 - val_side_accuracy: 0.8750 - val_side_loss: 0.3860 - learning_rate: 0.0010
Epoch 3/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step - loss: 2.0309 - sentiment_accuracy: 0.7414 - sentiment_loss: 0.6667 - severity_accuracy: 0.5542 - severity_loss: 1.0149 - side_accuracy: 0.8746 - side_loss: 0.3493

15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 299ms/step - loss: 1.7514 - sentiment_accuracy: 0.7750 - sentiment_loss: 0.5770 - severity_accuracy: 0.5917 - severity_loss: 0.9222 - side_accuracy: 0.9354 - side_loss: 0.2522 - val_loss: 1.3711 - val_sentiment_accuracy: 0.7667 - val_sentiment_loss: 0.5393 - val_severity_accuracy: 0.6250 - val_severity_loss: 0.7501 - val_side_accuracy: 0.9667 - val_side_loss: 0.0701 - learning_rate: 0.0010
Epoch 4/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - loss: 1.1067 - sentiment_accuracy: 0.8801 - sentiment_loss: 0.3617 - severity_accuracy: 0.6663 - severity_loss: 0.6947 - side_accuracy: 0.9862 - side_loss: 0.0503

...

15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 100ms/step - loss: 0.1425 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.0082 - severity_accuracy: 0.9771 - severity_loss: 0.1316 - side_accuracy: 1.0000 - side_loss: 0.0028 - val_loss: 0.4041 - val_sentiment_accuracy: 0.9917 - val_sentiment_loss: 0.0645 - val_severity_accuracy: 0.8667 - val_severity_loss: 0.3116 - val_side_accuracy: 0.9917 - val_side_loss: 0.0204 - learning_rate: 5.0000e-04
Epoch 17/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - loss: 0.1191 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.0074 - severity_accuracy: 0.9838 - severity_loss: 0.1093 - side_accuracy: 1.0000 - side_loss: 0.0024

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step - loss: 0.1087 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.0088 - severity_accuracy: 0.9875 - severity_loss: 0.0977 - side_accuracy: 1.0000 - side_loss: 0.0022 - val_loss: 0.3639 - val_sentiment_accuracy: 1.0000 - val_sentiment_loss: 0.0108 - val_severity_accuracy: 0.8750 - val_severity_loss: 0.3299 - val_side_accuracy: 0.9917 - val_side_loss: 0.0171 - learning_rate: 5.0000e-04
Epoch 18/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - loss: 0.0948 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.0066 - severity_accuracy: 0.9904 - severity_loss: 0.0860 - side_accuracy: 1.0000 - side_loss: 0.0022

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - loss: 0.0861 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.0069 - severity_accuracy: 0.9917 - severity_loss: 0.0771 - side_accuracy: 1.0000 - side_loss: 0.0021 - val_loss: 0.3978 - val_sentiment_accuracy: 0.9917 - val_sentiment_loss: 0.0373 - val_severity_accuracy: 0.8917 - val_severity_loss: 0.3349 - val_side_accuracy: 0.9917 - val_side_loss: 0.0181 - learning_rate: 5.0000e-04
Epoch 19/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - loss: 0.0629 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.0070 - severity_accuracy: 0.9945 - severity_loss: 0.0544 - side_accuracy: 1.0000 - side_loss: 0.0015

15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 100ms/step - loss: 0.0581 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.0067 - severity_accuracy: 0.9958 - severity_loss: 0.0497 - side_accuracy: 1.0000 - side_loss: 0.0018 - val_loss: 0.2754 - val_sentiment_accuracy: 1.0000 - val_sentiment_loss: 0.0109 - val_severity_accuracy: 0.9167 - val_severity_loss: 0.2544 - val_side_accuracy: 1.0000 - val_side_loss: 0.0058 - learning_rate: 5.0000e-04
Epoch 20/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - loss: 0.0427 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.0066 - severity_accuracy: 0.9982 - severity_loss: 0.0346 - side_accuracy: 1.0000 - side_loss: 0.0015

15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 99ms/step - loss: 0.0411 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.0061 - severity_accuracy: 0.9979 - severity_loss: 0.0335 - side_accuracy: 1.0000 - side_loss: 0.0015 - val_loss: 0.3292 - val_sentiment_accuracy: 0.9917 - val_sentiment_loss: 0.0356 - val_severity_accuracy: 0.9000 - val_severity_loss: 0.2784 - val_side_accuracy: 1.0000 - val_side_loss: 0.0082 - learning_rate: 5.0000e-04


In [19]:
model.save("model/drug_review/drug_model.h5")

os.makedirs("model/drug_review", exist_ok=True)

with open("model/drug_review/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("model/drug_review/le_sent.pkl", "wb") as f:
    pickle.dump(le_sent, f)

with open("model/drug_review/le_side.pkl", "wb") as f:
    pickle.dump(le_side, f)

with open("model/drug_review/le_sev.pkl", "wb") as f:
    pickle.dump(le_sev, f)

print("✅ Saved drug review model")

✅ Saved drug review model
